In [ ]:
%run maths.ipynb
%run shell_mesh_builder.ipynb

In [ ]:
def build_chamber_septa(
    theta,
    shell_mesh,
    aperture_points=32,
    chamber_spacing=0.65,
    septum_rings=8,
    septum_depth=0.35
):
    """
    Build curved internal chamber walls inside the shell.

    Conceptually, these are septa: internal partitions laid down at intervals
    as the shell grows. In a Nautilus-like shell, the animal moves forward into
    the newest body chamber, leaving older chambers behind, separated by curved
    walls.

    This function takes the existing shell surface mesh and inserts chamber
    walls at selected aperture rings.

    Each septum is built as a shallow concave bowl:

    - the outer edge matches an existing aperture ring
    - nested smaller rings fill the aperture
    - the centre is pushed slightly backward along the growth direction
    - the result is a curved chamber wall rather than a flat disc

    Chamber colouring is varied by growth age:

    - older inner chambers are darker
    - newer outer chambers are lighter

    :param theta: Spiral angle values, representing growth progression
    :param shell_mesh: Tuple returned by build_shell_mesh
    :param aperture_points: Number of points around each aperture ring
    :param chamber_spacing: Angular distance between successive chamber walls
    :param septum_rings: Number of nested rings used to fill each septum
    :param septum_depth: Strength of concavity in each chamber wall
    :return: Chamber mesh arrays Xc, Yc, Zc, Cc and triangle index arrays Ic, Jc, Kc
    """

    # Unpack the outer shell mesh.
    # X, Y, Z contain all the surface vertices for all aperture rings.
    # The remaining values are ignored here because chamber construction
    # only needs the shell geometry, not its colour or triangle faces.
    X, Y, Z, _, _, _, _ = shell_mesh

    # Number of growth stages / aperture rings in the shell.
    n_spiral = len(theta)

    # Chamber mesh vertices.
    Xc = []
    Yc = []
    Zc = []

    # Chamber colour/intensity values.
    Cc = []

    # Chamber triangle index arrays for Plotly Mesh3d.
    Ic = []
    Jc = []
    Kc = []

    # Track the theta value where the previous chamber was inserted.
    # New chamber walls are added once growth has advanced by chamber_spacing.
    last_chamber_theta = theta[0]

    # Walk along the shell growth path.
    # Each index s corresponds to one aperture ring in the outer shell mesh.
    for s in range(1, n_spiral - 1):

        # Add a chamber wall only at selected intervals.
        if theta[s] - last_chamber_theta >= chamber_spacing:

            # Locate the vertices belonging to the aperture ring at this
            # growth stage. This ring becomes the outer edge of the septum.
            start_index = s * aperture_points

            ring_x = X[start_index:start_index + aperture_points]
            ring_y = Y[start_index:start_index + aperture_points]
            ring_z = Z[start_index:start_index + aperture_points]

            # Estimate the centre of the aperture ring.
            # The septum will be built inward from this outer ring.
            centre_x = np.mean(ring_x)
            centre_y = np.mean(ring_y)
            centre_z = np.mean(ring_z)

            # -------------------------------------------------------------
            # Estimate local growth direction
            # -------------------------------------------------------------
            # To curve the septum, we need to know which way is "backward"
            # into the older part of the shell.
            #
            # We estimate the growth direction by comparing the mean position
            # of nearby aperture rings before and after the current one.
            if s > 0:
                dx = (
                    np.mean(X[(s+1)*aperture_points:(s+2)*aperture_points])
                    - np.mean(X[(s-1)*aperture_points:s*aperture_points])
                )
                dy = (
                    np.mean(Y[(s+1)*aperture_points:(s+2)*aperture_points])
                    - np.mean(Y[(s-1)*aperture_points:s*aperture_points])
                )
                dz = (
                    np.mean(Z[(s+1)*aperture_points:(s+2)*aperture_points])
                    - np.mean(Z[(s-1)*aperture_points:s*aperture_points])
                )
            else:
                dx, dy, dz = 0, 0, 1

            direction = np.array([dx, dy, dz])

            # Normalise the direction vector so it only represents direction,
            # not the distance between aperture rings.
            direction = direction / np.linalg.norm(direction)

            # Approximate aperture size.
            # This gives a scale for how deeply the septum should curve.
            aperture_radius = np.mean(
                np.sqrt(
                    (ring_x - centre_x)**2 +
                    (ring_y - centre_y)**2 +
                    (ring_z - centre_z)**2
                )
            )

            # Starting index of this septum"s vertices in the chamber mesh.
            base_index = len(Xc)

            # -------------------------------------------------------------
            # Build one curved septum from nested rings
            # -------------------------------------------------------------
            # t = 0 gives the centre of the septum.
            # t = 1 gives the original aperture ring.
            #
            # By creating intermediate rings, we can make a smooth curved
            # surface instead of a single flat fan of triangles.
            for ring_idx in range(septum_rings + 1):

                t = ring_idx / septum_rings

                # Bowl curve:
                # - deepest at the centre
                # - smoothly approaches zero at the aperture edge
                #
                # This pushes the septum inward toward older shell growth.
                curve_offset = -septum_depth * aperture_radius * (1 - t**2)

                for p in range(aperture_points):

                    # Interpolate between the centre and the aperture rim.
                    # This creates one nested ring within the septum.
                    px = centre_x + t * (ring_x[p] - centre_x)
                    py = centre_y + t * (ring_y[p] - centre_y)
                    pz = centre_z + t * (ring_z[p] - centre_z)

                    # Push the point backward along the growth direction,
                    # producing the shallow concavity of the chamber wall.
                    px += curve_offset * direction[0]
                    py += curve_offset * direction[1]
                    pz += curve_offset * direction[2]

                    Xc.append(px)
                    Yc.append(py)
                    Zc.append(pz)

                    # Colour by growth age.
                    # Inner/older chambers have smaller growth_fraction.
                    # Outer/newer chambers have larger growth_fraction.
                    growth_fraction = s / (n_spiral - 1)

                    # Older inner chambers darker, newer outer chambers lighter.
                    chamber_colour = 0.4 + 0.6 * growth_fraction

                    Cc.append(chamber_colour)

            # -------------------------------------------------------------
            # Stitch the nested septum rings into a curved surface
            # -------------------------------------------------------------
            # Each pair of neighbouring nested rings forms a band of the
            # chamber wall. Each band is split into triangles.
            for ring_idx in range(septum_rings):

                ring_a = base_index + ring_idx * aperture_points
                ring_b = base_index + (ring_idx + 1) * aperture_points

                for p in range(aperture_points):

                    a0 = ring_a + p
                    a1 = ring_a + ((p + 1) % aperture_points)
                    b0 = ring_b + p
                    b1 = ring_b + ((p + 1) % aperture_points)

                    # First triangle in this small septum patch.
                    Ic.append(a0)
                    Jc.append(b0)
                    Kc.append(a1)

                    # Second triangle in this small septum patch.
                    Ic.append(a1)
                    Jc.append(b0)
                    Kc.append(b1)

            # Record where this chamber was added so the next one is spaced
            # further along the shell"s growth path.
            last_chamber_theta = theta[s]

    return np.array(Xc), np.array(Yc), np.array(Zc), np.array(Cc), Ic, Jc, Kc

In [ ]:
def build_siphuncle_mesh(
    theta,
    shell_mesh,
    aperture_points=32,
    siphuncle_radius=0.6,
    siphuncle_offset=0.35,
    tube_points=16,
    chamber_spacing=None,
    siphuncle_end_fraction=0.98,
    include_terminal_ring=True
):
    """
    Build a simplified siphuncle running through the chambered part of the shell.

    Conceptually, the siphuncle is treated as a small tube threading through
    the internal chamber septa.

    In a chambered cephalopod shell, the siphuncle does not need to extend all
    the way to the open mouth of the shell. Instead, it should run through the
    chambered region and reach the final chamber wall near the start of the
    living body chamber.

    To keep the model simple and visually coherent, the siphuncle path is
    generated from aperture rings along the shell growth path:

    1. aperture ring centres are estimated from the shell mesh
    2. the siphuncle position is offset slightly from each centre
    3. selected positions are used to form a tube path
    4. neighbouring tube rings are stitched into a continuous mesh

    If chamber_spacing is supplied, siphuncle rings are placed at the same
    approximate interval as the chamber septa. This keeps the siphuncle visually
    aligned with the chambered structure rather than stopping too early or
    running arbitrarily through the body chamber.

    :param theta: Spiral angle values representing shell growth
    :param shell_mesh: Mesh returned by build_shell_mesh
    :param aperture_points: Number of vertices around each aperture ring
    :param siphuncle_radius: Radius of the siphuncle tube
    :param siphuncle_offset: Fractional offset from chamber centre
    :param tube_points: Number of vertices around the siphuncle tube
    :param chamber_spacing: Optional angular spacing matching chamber septa
    :param siphuncle_end_fraction: Fraction of shell growth where siphuncle stops
    :param include_terminal_ring: Whether to add a final ring at siphuncle_end_fraction
    :return: Xs, Ys, Zs vertex arrays and triangle index arrays Is, Js, Ks
    """

    X, Y, Z, _, _, _, _ = shell_mesh

    n_spiral = len(theta)

    # Angles around the circular siphuncle cross-section.
    phi = np.linspace(0, 2*np.pi, tube_points, endpoint=False)

    # -----------------------------------------------------------------
    # Choose which growth stages should carry siphuncle tube rings
    # -----------------------------------------------------------------
    # The siphuncle should run through the chambered region of the shell,
    # not necessarily all the way to the final open aperture.
    #
    # siphuncle_end_fraction therefore marks the approximate boundary
    # between the chambered shell and the final body chamber / aperture lip.
    max_growth_index = int((n_spiral - 1) * siphuncle_end_fraction)
    max_growth_index = max(1, min(max_growth_index, n_spiral - 1))

    siphuncle_indices = []

    if chamber_spacing is None:
        # Fallback behaviour:
        # use every aperture ring up to the end of the chambered region.
        # This gives the smoothest tube, but it is less explicitly tied to
        # the chamber septa.
        siphuncle_indices = list(range(max_growth_index + 1))

    else:
        # Chamber-aligned behaviour:
        # place siphuncle rings at roughly the same theta intervals used
        # by the chamber septa builder. This makes the siphuncle read as
        # a structure passing from chamber to chamber.
        last_siphuncle_theta = theta[0]
        siphuncle_indices.append(0)

        for s in range(1, max_growth_index + 1):
            if theta[s] - last_siphuncle_theta >= chamber_spacing:
                siphuncle_indices.append(s)
                last_siphuncle_theta = theta[s]

        # Add a final terminal ring near the body-chamber boundary.
        # This prevents the siphuncle from appearing to stop well before
        # the final chamber if the chamber spacing does not land exactly
        # on siphuncle_end_fraction.
        if include_terminal_ring and siphuncle_indices[-1] != max_growth_index:
            siphuncle_indices.append(max_growth_index)

    Xs = []
    Ys = []
    Zs = []

    # -----------------------------------------------------------------
    # Build one siphuncle tube ring at each selected chamber position
    # -----------------------------------------------------------------
    for s in siphuncle_indices:

        # Locate the aperture ring corresponding to this growth stage.
        start_index = s * aperture_points

        ring_x = X[start_index:start_index + aperture_points]
        ring_y = Y[start_index:start_index + aperture_points]
        ring_z = Z[start_index:start_index + aperture_points]

        # Estimate the centre of the aperture ring.
        # This gives the local chamber centreline before applying the
        # siphuncle offset.
        centre_x = np.mean(ring_x)
        centre_y = np.mean(ring_y)
        centre_z = np.mean(ring_z)

        # Estimate local shell radius in the coiling plane.
        # This gives a scale for moving the siphuncle away from the exact
        # geometric centre of the chamber.
        radius = np.mean(
            np.sqrt(
                (ring_x - centre_x)**2 +
                (ring_y - centre_y)**2
            )
        )

        # -------------------------------------------------------------
        # Offset the siphuncle away from the chamber centre.
        # -------------------------------------------------------------
        # Real siphuncles are typically offset rather than passing exactly
        # through the chamber centre. This simplified offset uses the
        # spiral angle to keep the siphuncle position rotating with the
        # growth path.
        sx = centre_x + siphuncle_offset * radius * np.cos(theta[s])
        sy = centre_y + siphuncle_offset * radius * np.sin(theta[s])
        sz = centre_z

        # -------------------------------------------------------------
        # Generate one circular siphuncle ring.
        # -------------------------------------------------------------
        # Each ring is a small circular cross-section. Connecting these
        # rings creates the visible tube.
        for p in phi:

            local_x = siphuncle_radius * np.cos(p)
            local_z = siphuncle_radius * np.sin(p)

            px = sx + local_x
            py = sy
            pz = sz + local_z

            Xs.append(px)
            Ys.append(py)
            Zs.append(pz)

    Xs = np.array(Xs)
    Ys = np.array(Ys)
    Zs = np.array(Zs)

    Is = []
    Js = []
    Ks = []

    # -----------------------------------------------------------------
    # Stitch neighbouring siphuncle rings into a continuous tube.
    # -----------------------------------------------------------------
    # Each pair of neighbouring siphuncle rings forms a short tube segment.
    # Each quad between them is split into two triangles for Plotly Mesh3d.
    n_siphuncle_rings = len(siphuncle_indices)

    for s in range(n_siphuncle_rings - 1):
        for p in range(tube_points):

            current = s * tube_points + p
            next_p = s * tube_points + ((p + 1) % tube_points)

            current_next = (s + 1) * tube_points + p
            next_next = (s + 1) * tube_points + ((p + 1) % tube_points)

            Is.append(current)
            Js.append(current_next)
            Ks.append(next_p)

            Is.append(next_p)
            Js.append(current_next)
            Ks.append(next_next)

    return Xs, Ys, Zs, Is, Js, Ks

In [ ]:
import json

def load_options(options_file):
    """
    Load a JSON-formatted options file that acts as a "shell preset".
    
    The options file keeps the editable model parameters separate from the mesh-building
    code, making it easier to define different shell forms such as Nautilus-like,
    ammonite-like or other exploratory morphologies.

    :param options_file: Path to the options file
    :return: Dictionary of values read from the options file
    """
    # Load the options file.
    with open(options_file) as f:
        return json.load(f)

In [ ]:
def build_shell_components(options):
    """
    Build the 3D meshes for the shell, chamber septa and siphuncle from an options file.

    The JSON options file acts as a shell preset. It keeps the editable model
    param

    :param options: Dictionary of options including mesh building options
    :return: Tuple of the 3D mesh coordinates for the shell, chamber septa and siphuncle
    """
    # Generate the log spiral coordinates.
    # This defines the growth path followed by the shell aperture.
    spiral_options = options["spiral"]
    shell_options = options["shell"]
    if spiral_options["aperture_mode"] == "abutting":
        theta, r, x, y, width, height = logarithmic_spiral(
            a=spiral_options["a"],
            b=spiral_options["b"],
            turns=spiral_options["turns"],
            points=spiral_options["points"],
            aperture_mode="abutting",
            abutment_scale=spiral_options["abutment_scale"],
            abutment_height_ratio=spiral_options["abutment_height_ratio"]
        )
    else:
        theta, r, x, y, width, height = logarithmic_spiral(
            a=spiral_options["a"],
            b=spiral_options["b"],
            turns=spiral_options["turns"],
            points=spiral_options["points"],
            aperture_mode="proportional",
            aperture_width_multiplier=spiral_options["aperture_width_multiplier"],
            aperture_height_multiplier=spiral_options["aperture_height_multiplier"]
        )

    if spiral_options["growth_mode"] == "conical_helix":
        theta, r, x, y, z_path, width, height = conical_helix(
            a=spiral_options["a"],
            b=spiral_options["b"],
            turns=spiral_options["turns"],
            points=spiral_options["points"],
            spire_height=spiral_options["spire_height"],
            cone_taper=spiral_options["cone_taper"],
            aperture_width_multiplier=spiral_options["aperture_width_multiplier"],
            aperture_height_multiplier=spiral_options["aperture_height_multiplier"]
        )
    elif shell_options["automatic_z_path"]:
        # ---------------------------------------------------------------------
        # Optional automatic Z path for whorl abutment
        # ---------------------------------------------------------------------
        # Derive vertical rise from aperture height. This helps high-spired
        # gastropods form touching/overlapping whorls rather than separated
        # screw-like coils.

        # whorl_abutment < 1.0  : Slight overlap between whorls
        # whorl_abutment <= 1.0 : Whorls just touching
        # whorl_abutment > 1.0  : Gaps between whorls
        pitch_per_turn = height * shell_options["whorl_abutment"]

        dtheta = np.gradient(theta)
        dz = (pitch_per_turn / (2 * np.pi)) * dtheta

        z_path = np.cumsum(dz)

        # Start at zero for tidier positioning
        z_path -= z_path[0]
    else:
        z_path = None


    # Generate the 3D shell mesh.
    # This builds the outer shell surface, optional inner wall surface,
    # pigmentation, growth ribs and final aperture lip.
    shell_mesh = build_shell_mesh(
        theta,
        r,
        x,
        y,
        width,
        height,
        z_scale=shell_options["z_scale"],
        z_path=z_path,
        aperture_points=shell_options["aperture_points"],
        aperture_tilt=shell_options["aperture_tilt"],
        ribbing_amplitude=shell_options["ribbing_amplitude"],
        ribbing_frequency=shell_options["ribbing_frequency"],
        lip_start=shell_options["lip_start"],
        lip_strength=shell_options["lip_strength"],
        lip_power=shell_options["lip_power"],
        wall_enabled=shell_options["wall_enabled"],
        wall_thickness=shell_options["wall_thickness"],
        inner_surface_colour_shift=shell_options["inner_surface_colour_shift"]
    )

    # Generate the septa separating the chambers.
    # These are the curved internal walls left behind as the shell grows.
    chamber_options = options["chambers"]
    if chamber_options["enabled"]:
        chamber_mesh = build_chamber_septa(
            theta,
            shell_mesh,
            aperture_points=chamber_options["aperture_points"],
            chamber_spacing=chamber_options["chamber_spacing"],
            septum_rings=chamber_options["septum_rings"],
            septum_depth=chamber_options["septum_depth"]
        )
    else:
        chamber_mesh = None

    # Generate the siphuncle.
    # The siphuncle is tied to the chambered region of the shell. By default,
    # it uses the same spacing as the chamber septa and stops near the start
    # of the final body chamber / lip region, rather than running all the way
    # to the shell mouth.
    siphuncle_options = options["siphuncle"]
    if siphuncle_options["enabled"]:
        siphuncle_mesh = build_siphuncle_mesh(
            theta,
            shell_mesh,
            aperture_points=siphuncle_options.get(
                "aperture_points",
                shell_options["aperture_points"]
            ),
            siphuncle_radius=siphuncle_options["siphuncle_radius"],
            siphuncle_offset=siphuncle_options["siphuncle_offset"],
            tube_points=siphuncle_options["tube_points"],
            chamber_spacing=siphuncle_options.get(
                "chamber_spacing",
                chamber_options.get("chamber_spacing")
            ),
            siphuncle_end_fraction=siphuncle_options.get(
                "siphuncle_end_fraction",
                shell_options.get("lip_start", 0.90)
            ),
            include_terminal_ring=siphuncle_options.get(
                "include_terminal_ring",
                True
            )
        )
    else:
        siphuncle_mesh = None

    return shell_mesh, chamber_mesh, siphuncle_mesh


In [ ]:
import plotly.graph_objects as go


def plot_shell_mesh(
    options,
    shell_mesh,
    chamber_mesh=None,
    siphuncle_mesh=None
):
    """
    Render the procedural shell geometry using Plotly Mesh3d.

    Conceptually, this function visualises two interacting structures:

    1. The outer shell surface
       - built from a sequence of connected aperture rings
       - representing the accumulated shell deposited during growth
       - optionally including an inner wall surface and visible aperture rim

    2. Optional internal chamber septa
       - curved internal walls inserted periodically during growth
       - representing the compartmentalised interior of a cephalopod shell

    The shell surface and internal chambers are rendered as separate meshes,
    allowing transparency and lighting to reveal the shell"s internal
    architecture. If shell wall thickness is enabled, the shell mesh already
    contains both the outer surface and the inner wall surface.

    Colour values are interpreted as shell pigmentation patterns generated
    during growth.

    Transparency can be used to create:
    - opaque shell reconstructions
    - translucent museum-style cutaways
    - exposed chamber visualisations

    :param options: Dictionary of plotting options
    :param shell_mesh: Tuple returned by build_shell_mesh
    :param chamber_mesh: Optional chamber mesh returned by build_chamber_septa
    :param siphuncle_mesh: Optional siphuncle mesh returned by build_siphuncle_mesh
    """

    # ---------------------------------------------------------------------
    # Unpack the outer shell mesh
    # ---------------------------------------------------------------------
    #
    # X, Y, Z:
    #   vertex positions for the shell surface
    #
    # C:
    #   per-vertex colour/intensity values used for shell pigmentation
    #
    # I, J, K:
    #   triangle index arrays defining the shell surface geometry
    X, Y, Z, C, I, J, K = shell_mesh

    # ---------------------------------------------------------------------
    # Create the outer shell surface
    # ---------------------------------------------------------------------
    #
    # Plotly Mesh3d renders the shell as many connected triangles.
    #
    # The shell colour comes from the generated pigmentation bands.
    #
    # Lighting parameters are tuned to give a nacreous / shell-like sheen
    # rather than a matte engineering-style surface.
    traces = [
        go.Mesh3d(
            x=X,
            y=Y,
            z=Z,

            # Triangle vertex indices
            i=I,
            j=J,
            k=K,

            # Procedural shell pigmentation
            intensity=C,
            colorscale="YlOrBr",
            showscale=False,

            # Transparency allows internal chambers to be revealed
            opacity=options["shell_opacity"],

            # Smooth shading produces a more organic shell surface
            flatshading=False,

            # Lighting tuned for shell-like highlights and soft reflections
            lighting=dict(
                ambient=0.35,
                diffuse=0.95,
                specular=1.2,
                roughness=0.12,
                fresnel=0.5
            )
        )
    ]

    # ---------------------------------------------------------------------
    # Add the shell mesh wireframe, if requested
    # ---------------------------------------------------------------------
    if options["show_wireframe"]:
        # ---------------------------------------------------------------------
        # Construct a wireframe representation of the shell mesh
        # ---------------------------------------------------------------------
        #
        # The shell surface is stored as a triangle mesh.
        #
        # Each triangle is defined by three vertex indices:
        #
        #     (i, j, k)
        #
        # For example:
        #
        #     i = 42
        #     j = 43
        #     k = 58
        #
        # represents a triangle connecting vertices:
        #
        #     42 ----- 43
        #       \     /
        #        \   /
        #          58
        #
        # A wireframe view requires the set of unique edges in the mesh:
        #
        #     (42,43)
        #     (43,58)
        #     (58,42)
        #
        # Because neighbouring triangles share edges, many edges appear twice
        # in the triangle list.
        #
        # Example:
        #
        #     Triangle A -> (42,43,58)
        #     Triangle B -> (43,42,61)
        #
        # Both triangles contain the same geometric edge:
        #
        #     42 ----- 43
        #
        # but one triangle references it as:
        #
        #     (42,43)
        #
        # and the other as:
        #
        #     (43,42)
        #
        # To ensure both representations are recognised as the same edge, the
        # vertex pair is sorted before storage:
        #
        #     (43,42) -> (42,43)
        #
        # A Python set is then used to automatically eliminate duplicates,
        # leaving only one copy of each unique edge in the mesh.
        #
        # The resulting edge set forms the topological skeleton of the shell
        # and can be rendered as a wireframe overlay.
        # ---------------------------------------------------------------------

        edges = set()

        for i, j, k in zip(I, J, K):

            # Add the three edges belonging to this triangle.
            #
            # sorted() ensures edge direction is ignored:
            #
            #     (42,43) == (43,42)
            #
            # once converted to:
            #
            #     (42,43)
            #
            edges.add(tuple(sorted((i, j))))
            edges.add(tuple(sorted((j, k))))
            edges.add(tuple(sorted((k, i))))

        # ---------------------------------------------------------------------
        # Convert unique edge pairs into Plotly line coordinates
        # ---------------------------------------------------------------------
        #
        # Scatter3d expects explicit coordinate sequences.
        #
        # For each edge:
        #
        #     (a,b)
        #
        # we look up the corresponding vertex positions:
        #
        #     (X[a], Y[a], Z[a])
        #     (X[b], Y[b], Z[b])
        #
        # and append them to the plotting arrays.
        #
        # Plotly uses None as a line separator.
        #
        # Example:
        #
        #     [P1, P2, None, P3, P4, None]
        #
        # is interpreted as:
        #
        #     Line 1 : P1 -> P2
        #     Line 2 : P3 -> P4
        #
        # rather than one continuous polyline.
        #
        # The resulting coordinate arrays therefore describe many independent
        # line segments, one for each unique mesh edge.
        # ---------------------------------------------------------------------

        edge_x = []
        edge_y = []
        edge_z = []

        for a, b in edges:

            edge_x.extend([
                X[a],      # Start vertex x-coordinate
                X[b],      # End vertex x-coordinate
                None       # End this line segment
            ])

            edge_y.extend([
                Y[a],
                Y[b],
                None
            ])

            edge_z.extend([
                Z[a],
                Z[b],
                None
            ])

        # ---------------------------------------------------------------------
        # Add the wireframe as a Plotly Scatter3d trace
        # ---------------------------------------------------------------------
        #
        # mode="lines" causes Plotly to draw each edge segment.
        #
        # When combined with a semi-transparent shell surface this provides a
        # useful debugging view that reveals:
        #
        # - mesh density
        # - triangle orientation
        # - aperture ring structure
        # - shell growth progression
        # - topological artefacts and gaps
        #
        # It can also be used as a standalone visualisation by hiding the
        # filled shell mesh entirely.
        # ---------------------------------------------------------------------

        traces.append(
            go.Scatter3d(
                x=edge_x,
                y=edge_y,
                z=edge_z,

                mode="lines",

                line=dict(
                    color="black",
                    width=1
                ),

                showlegend=False
            )
        )

    # ---------------------------------------------------------------------
    # Add internal chamber septa if present
    # ---------------------------------------------------------------------
    #
    # The chamber mesh is rendered separately from the shell surface so
    # transparency and colouring can differ between shell wall and chambers.
    if chamber_mesh is not None:

        # Chamber mesh vertices, colours and triangle indices
        Xc, Yc, Zc, Cc, Ic, Jc, Kc = chamber_mesh

        traces.append(
            go.Mesh3d(
                x=Xc,
                y=Yc,
                z=Zc,

                i=Ic,
                j=Jc,
                k=Kc,

                # Chamber colour/intensity values.
                # Older chambers can be darker than newer ones.
                intensity=Cc,

                colorscale="YlOrBr",

                # Chamber transparency helps preserve visibility through
                # translucent outer shell walls.
                opacity=options["septa_opacity"],

                flatshading=False,

                showscale=False
            )
        )

    # ---------------------------------------------------------------------
    # Add the siphuncle if present
    # ---------------------------------------------------------------------
    if siphuncle_mesh is not None:
        Xs, Ys, Zs, Is, Js, Ks = siphuncle_mesh
        traces.append(
            go.Mesh3d(
            x=Xs,
            y=Ys,
            z=Zs,
            i=Is,
            j=Js,
            k=Ks,
            color="rgb(180,120,80)",
            opacity=options["siphuncle_opacity"],
            flatshading=False,
            showscale=False)
        )

    # ---------------------------------------------------------------------
    # Assemble the full figure
    # ---------------------------------------------------------------------
    fig = go.Figure(data=traces)

    # ---------------------------------------------------------------------
    # Configure the viewing scene
    # ---------------------------------------------------------------------
    #
    # aspectmode="data" preserves the shell proportions correctly.
    #
    # Without this, Plotly may distort the shell geometry to fit the figure.
    if options["hide_axes"]:
        fig.update_layout(
            title=dict(
                text=options["title"],
                x=0.5,
                xanchor='center'
            ),
            scene=dict(
                xaxis=dict(visible=False),
                yaxis=dict(visible=False),
                zaxis=dict(visible=False),
                bgcolor='rgba(0,0,0,0)',

                # Preserve true shell proportions
                aspectmode="data"
            ),
            margin=dict(l=0, r=0, t=0, b=0),
            paper_bgcolor='rgba(0,0,0,1)',
            plot_bgcolor='rgba(0,0,0,1)',
            width=options["width"],
            height=options["height"]
        )
    else:
        fig.update_layout(
            title=dict(
                text=options["title"],
                x=0.5,
                xanchor='center'
            ),
            scene=dict(
                xaxis_title="x",
                yaxis_title="y",
                zaxis_title="z",

                # Preserve true shell proportions
                aspectmode="data"
            ),

            width=options["width"],
            height=options["height"]
        )

    # Display the interactive shell visualisation.
    #
    # The resulting figure can be:
    # - rotated
    # - zoomed
    # - inspected interactively
    #
    # revealing both external morphology and internal chamber structure.
    fig.show()

In [ ]:
from pathlib import Path
import os

def main(options_file_name):
    """
    Entry point for computing the shell's internal structure and surface

    :param options_file_name: Name of the options file to use, excluding path and extension
    """

    # Load the options file
    options_file_path = Path(os.getcwd()).parent / "data" / f"{options_file_name}.json"
    options = load_options(options_file_path)

    # Build the meshes describing the shell's features
    shell_mesh, chamber_mesh, siphuncle_mesh = build_shell_components(options)

    # Plot the shell with an opaque surface
    opaque_plotting_options = options["plotting"]["opaque"]
    if opaque_plotting_options["enabled"]:
        plot_shell_mesh(opaque_plotting_options, shell_mesh, chamber_mesh, siphuncle_mesh)

    # Plot the shell with a transparent surface to reveal internal structure
    transparent_plotting_options = options["plotting"]["transparent"]
    if transparent_plotting_options["enabled"]:
        plot_shell_mesh(transparent_plotting_options, shell_mesh, chamber_mesh, siphuncle_mesh)


In [ ]:
NAME="ammonite"
main(NAME)